In [ ]:
import torch
import torch.nn as nn

#Prototype 1

##Functions

###MFs

In [ ]:
gaussian2_params = ['sigma', 'mu']
def gaussian2(x, p):
    return torch.exp(torch.pow((x - p[:, :, 0]), 2) / (2 * torch.pow(p[:, :, 1], 2)))

In [ ]:
gaussian3_params = ['sigma', 'mu', 'f']
def gaussian3(x, p):
    return torch.exp(-p[:, :, 2] * torch.pow((x - p[:, :, 0]), 2) / (2 * torch.pow(p[:, :, 1], 2)))

In [ ]:
bell_params = ["a", "b", "c"]
def bell(x, p):
    return 1 / (1 + torch.pow(torch.abs((x - p[:, :, 2]) / p[:, :, 0]), 2 * p[:, :, 1]))

In [ ]:
triangular_params = ["a", "b"]
def triangular(x, p):
    return torch.clamp(1 - torch.abs((x - p[:, :, 1]) / p[:, :, 0]), min=0)

In [ ]:
trapezoidal_params = ["a", "b", "c", "d"]
def trapezoidal(x, p):
    return torch.min(torch.clamp((x - p[:, :, 0]) / (p[:, :, 1] - p[:, :, 0]), min=0, max=1), torch.clamp((p[:, :, 3] - x) / (p[:, :, 3] - p[:, :, 2]), min=0, max=1))

###Consequent Functions

In [ ]:
def weighted_linear(x, c, w):
    return (x.matmul(c[:, :-1].transpose(0, 1)) + c[:, -1]).mul(w)

###Other Functions

In [ ]:
def sum(x):
    return torch.sum(x, dim=-1)

###Layers

In [ ]:
class FuzzifyLayer(nn.Module):
    '''
    Class that represents the first layer (fuzzification layer) of an ANFIS

    Attributes:
        input_size : size of a single input
        rules : number of initial rules
        mf: function used as membership function
        params: list with parameter names
        premises: array with the parameters used in each node of this layer
    '''
    def __init__(self, input_size, rules=3, mf=gaussian3, params=['sigma', 'mu', 'f']):
        super(FuzzifyLayer, self).__init__()
        self.input_size = input_size
        self.rules = rules
        self.mf = mf
        self.params = params
        self.premises = torch.rand(input_size, rules, len(params))

    def forward(self, x):
        return self.mf(x.unsqueeze(x.dim()), self.premises)

    #For both gaussian_3 and gaussian_2 function
    def uniform_premises(self, x_train):
        if self.rules > 1:
            min = torch.min(x_train, dim=0).values
            max = torch.max(x_train, dim=0).values
            stp = (max - min) / (self.rules - 1)
            for i in range(self.input_size):
                h = torch.arange(mi[i], ma[i] + stp[i], stp[i])
                for j in range(self.rules):
                    self.premises[i][j][0] = h[j]
                    self.premises[i][j][1] = stp[i]/2
                    if (len(self.params) == 3): self.premises[i][j][2] = 2
        else:
            for i in range(self.input_size):
                for j in range(self.rules):
                    self.premises[i][j][0] = torch.std(x_train[:, i])
                    self.premises[i][j][1] = torch.mean(x_train[:, i])
                    if (len(self.params) == 3): self.premises[i][j][2] = 2

    @property
    def premises_structure(self):
        print("Premises Structure:")
        for i in range(self.input_size):
            print(f"    x{i} parameters:")
            for j in range(self.rules):
                print(f"        rule {j + 1}:")
                [print(f"            {self.params[k]}: {self.premises[i, j, k]}") for k in range(len(self.params))]


In [ ]:
class FiringLevelsLayer(nn.Module):
    '''
    Class used to calculate firing levels and normalize them
    '''
    def __init__(self):
        super(FiringLevelsLayer, self).__init__()

    def forward(self, x):
        return torch.nn.functional.normalize(x.prod(dim=x.dim()-2), p=1, dim=-1)

In [ ]:
class ConsequentLayer(nn.Module):
    '''
    Class that represents the fourth layer (consequent layer) of an ANFIS

    Attributes:
        input_size : size of a single input
        rules : number of rules
        function : function used as a consequent function
        consequents: array with the parameters used in each node of this layer
    '''
    def __init__(self, input_size, rules=3, function=weighted_linear):
        super(ConsequentLayer, self).__init__()
        self.rules = rules
        self.input_size = input_size
        self.function = function
        #CONSEQUENT PARAMETERS randomly initialized at the moment
        self.consequents = torch.rand(rules, input_size + 1)

    def forward(self, x, w):
        return self.function(x, self.consequents, w)

    @property
    def consequents_structure(self):
        print("Consequents Structure:")
        for i in range(self.rules):
            print(f"    rule {i + 1} consequent parameters: {self.consequents[i]}")

In [ ]:
class OutputLayer(nn.Module):
    '''
    Class that represents the fifth layer (output layer) of an ANFIS
    '''
    def __init__(self, function=sum):
        super(OutputLayer, self).__init__()
        self.function = function

    def forward(self, x):
        return self.function(x)

##Strucuture

In [ ]:
class Type3ANFIS(nn.Module):
    '''
    Class that represents a type3 ANFIS

    To initialize it:
        input_size : size of a single input
        init_rules : number of initial rules
        cf: function used as consequent function
        mf: function used as membership function
        of: function used to join the outputs of each of the rules
        mf_params: list with MF parameters namesr
        x_train: training data set; this parameter its used to initialize the premises uniformly,
                 if it is not entered, the premises will be initialized randomly
    '''
    def __init__(self, input_size, init_rules=3, cf=weighted_linear, mf=gaussian3, of=sum, mf_params=['sigma', 'mu', 'f'], prem_init_mode=0, x_train=[]):
        super(Type3ANFIS, self).__init__()
        self.fuzzify_layer = FuzzifyLayer(input_size, init_rules, mf, mf_params)
        if x_train != []:
            fuzzify_layer.uniform_premises(x_train)
        self.firing_levels_layer = FiringLevelsLayer()
        self.consequent_layer = ConsequentLayer(input_size, init_rules, cf)
        self.output_layer = OutputLayer(of)

    def forward(self, x):
        output = self.fuzzify_layer(x)
        output = self.consequent_layer(x, self.firing_levels_layer(output))
        output = self.output_layer(x)
        return output